In [2]:
#how does proximity to financial services, job density, population density, education level, and employment affect household income.

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
import scipy.stats as st
import statsmodels.api as sm 
import pylab as py 

# here are some of the tools we will use for our analyses
from sklearn.linear_model import LinearRegression
from sklearn.metrics import PredictionErrorDisplay
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score


import itertools
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor


from functools import partial
from sklearn.model_selection import \
     (cross_validate,
      KFold,
      ShuffleSplit)
from sklearn.base import clone
from ISLP.models import sklearn_sm


import pandas as pd
import csv
import sqlite3


In [3]:
#Classification model starts here

In [10]:
income = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Household Income.csv", na_values="NA")
prox_bank = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Prox_bank_csv.csv", na_values="NA")
employment = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Employment.csv", na_values="NA")
job_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Job_Density_csv.csv", na_values="NA")
pop_density = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Population_Density_csv.csv", na_values="NA")
education = pd.read_csv(r"C:\Users\addis\Downloads\DTSC\Education Level - Bachelor's Degree.csv", na_values="NA")

In [11]:
income.columns

Index(['NPA', '2023'], dtype='object')

In [12]:
prox_bank.columns

Index(['NPA', '2023'], dtype='object')

In [13]:
employment.columns

Index(['NPA', '2023'], dtype='object')

In [14]:
job_density.columns

Index(['NPA', '2022'], dtype='object')

In [15]:
pop_density.columns

Index(['NPA', '2020'], dtype='object')

In [16]:
education.columns

Index(['NPA', '2023'], dtype='object')

In [17]:
conn = sqlite3.connect("my_data.db")

income.to_sql("income", conn, if_exists="replace", index=False)
prox_bank.to_sql("prox_bank", conn, if_exists="replace", index=False)
employment.to_sql("employment", conn, if_exists="replace", index=False)
job_density.to_sql("job_density", conn, if_exists="replace", index=False)
pop_density.to_sql("pop_density", conn, if_exists="replace", index=False)
education.to_sql("education", conn, if_exists="replace", index=False) 

459

In [18]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE income(
    NPA INT,
    median_income DOUBLE
)
""")

cursor.execute("""
CREATE TABLE prox_bank(
    NPA INT,
    "2023" DOUBLE
)
""")

cursor.execute("""
CREATE TABLE employment(
    NPA INT,
    employment_rate DOUBLE
)
""")

cursor.execute("""
CREATE TABLE job_density(
    NPA INT,
    "2022" INT
)
""")

cursor.execute("""
CREATE TABLE pop_density(
    NPA INT,
    "2020" INT
)
""")

cursor.execute("""
CREATE TABLE education(
    NPA INT,
    bachelors_percent DOUBLE
)
""")

In [19]:
for _, row in income.iterrows():
    cursor.execute("INSERT INTO income VALUES (?, ?)",
                   (row['NPA'], row['2023']))

for _, row in prox_bank.iterrows():
    cursor.execute("INSERT INTO prox_bank VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

for _, row in employment.iterrows():
    cursor.execute("INSERT INTO employment VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

for _, row in job_density.iterrows():
    cursor.execute("INSERT INTO job_density VALUES (?, ?)", 
                   (row['NPA'], row['2022']))

for _, row in pop_density.iterrows():
    cursor.execute("INSERT INTO pop_density VALUES (?, ?)", 
                   (row['NPA'], row['2020']))

for _, row in education.iterrows():
    cursor.execute("INSERT INTO education VALUES (?, ?)", 
                   (row['NPA'], row['2023']))

In [24]:
query = """
SELECT 
    i.NPA,
    i."2023" AS income,
    pb."2023" AS prox_bank,
    e."2023" AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed."2023" AS bachelors_percent
FROM income i
LEFT JOIN prox_bank pb ON i.NPA = pb.NPA
LEFT JOIN employment e ON i.NPA = e.NPA
LEFT JOIN job_density jd ON i.NPA = jd.NPA
LEFT JOIN pop_density pd ON i.NPA = pd.NPA
LEFT JOIN education ed ON i.NPA = ed.NPA

UNION

SELECT 
    pb.NPA,
    i."2023" AS income,
    pb."2023" AS prox_bank,
    e."2023" AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed."2023" AS bachelors_percent
FROM prox_bank pb
LEFT JOIN income i ON pb.NPA = i.NPA
LEFT JOIN employment e ON pb.NPA = e.NPA
LEFT JOIN job_density jd ON pb.NPA = jd.NPA
LEFT JOIN pop_density pd ON pb.NPA = pd.NPA
LEFT JOIN education ed ON pb.NPA = ed.NPA
"""

In [33]:
query = """
SELECT 
    i.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent
FROM income i
LEFT JOIN prox_bank pb ON i.NPA = pb.NPA
LEFT JOIN employment e ON i.NPA = e.NPA
LEFT JOIN job_density jd ON i.NPA = jd.NPA
LEFT JOIN pop_density pd ON i.NPA = pd.NPA
LEFT JOIN education ed ON i.NPA = ed.NPA

UNION

SELECT 
    pb.NPA,
    i.median_income AS income,
    pb."2023" AS prox_bank,
    e.employment_rate AS employment,
    jd."2022" AS job_density,
    pd."2020" AS pop_density,
    ed.bachelors_percent AS bachelors_percent
FROM prox_bank pb
LEFT JOIN income i ON pb.NPA = i.NPA
LEFT JOIN employment e ON pb.NPA = e.NPA
LEFT JOIN job_density jd ON pb.NPA = jd.NPA
LEFT JOIN pop_density pd ON pb.NPA = pd.NPA
LEFT JOIN education ed ON pb.NPA = ed.NPA

"""

result = pd.read_sql_query(query, conn)
print(result.head())

Empty DataFrame
Columns: [NPA, income, prox_bank, employment, job_density, pop_density, bachelors_percent]
Index: []
